In [10]:
import joblib
import pandas as pd
import numpy as np
import warnings
import json

warnings.filterwarnings('ignore')

def load_artifacts():
    """Loads the trained model pipeline, feature names, and metadata."""
    try:
        model = joblib.load('final_model_pipeline.joblib')
        features = joblib.load('feature_names.joblib')
        with open('model_metadata.json', 'r') as f:
            metadata = json.load(f)
        return model, features, metadata
    except Exception as e:
        print(f"Error loading artifacts: {e}")
        return None, None, None

def validate_input(input_df, required_features):
    """Validates that the input data matches the exact requirements."""
    # Check for missing features
    missing_features = [f for f in required_features if f not in input_df.columns]
    if missing_features:
        raise ValueError(f"Missing required features: {missing_features}")
        
    # Check for extra features
    extra_features = [f for f in input_df.columns if f not in required_features]
    if extra_features:
        raise ValueError(f"Unexpected extra features: {extra_features}")
        
    # Check for NaNs
    if input_df.isnull().any().any():
        raise ValueError("Input data contains missing values (NaNs). Please provide complete data.")
        
    # Check for non-numeric data
    if not all(pd.api.types.is_numeric_dtype(input_df[col]) for col in input_df.columns):
        raise ValueError("Input data contains non-numeric values.")
        
    # Check for infinite values
    if np.isinf(input_df.values).any():
        raise ValueError("Input data contains infinite values.")
        
    # Reorder exactly to match training
    return input_df[required_features]

def predict_tumor(input_data):
    """
    Predicts whether a tumor is benign or malignant using strict validation.
    
    Args:
        input_data (dict or pd.DataFrame): The input features. Must match the feature names used during training exactly.
        
    Returns:
        dict: A dictionary containing the prediction, probability, and threshold used.
    """
    model, features, metadata = load_artifacts()
    if model is None:
        return {"error": "Model or artifacts not loaded"}
    
    # Ensure input_data is a DataFrame
    if isinstance(input_data, dict):
        input_data_df = pd.DataFrame([input_data])
    elif isinstance(input_data, pd.DataFrame):
        input_data_df = input_data.copy()
    else:
        return {"error": "Input must be a dictionary or pandas DataFrame"}

    try:
        input_data_df = validate_input(input_data_df, features)
    except ValueError as e:
        return {"error": str(e)}

    # Use the optimized threshold determined during evaluation
    optimal_threshold = metadata.get('optimal_threshold', 0.5)
    
    if hasattr(model.named_steps['model'], "predict_proba"):
        probabilities = model.predict_proba(input_data_df)[:, 1]
        score_name = "Probability"
    elif hasattr(model.named_steps['model'], "decision_function"):
        probabilities = model.decision_function(input_data_df)
        score_name = "Decision Score"
    else:
        probabilities = model.predict(input_data_df)
        score_name = "Prediction Score"
        
    prob_malignant = probabilities[0]
    
    # Apply custom threshold
    prediction_class = 1 if prob_malignant >= optimal_threshold else 0
    prediction_label = "Malignant" if prediction_class == 1 else "Benign"
    
    return {
        "Prediction": prediction_label,
        "Score": float(prob_malignant),
        "Score Type": score_name,
        "Threshold": optimal_threshold
    }

if __name__ == "__main__":
    # Example prediction using a synthetic Malignant-like sample
    sample = {
        'radius_mean': 20.57,
        'texture_mean': 17.77,
        'perimeter_mean': 132.9,
        'area_mean': 1326.0,
        'smoothness_mean': 0.08474,
        'compactness_mean': 0.07864,
        'concavity_mean': 0.0869,
        'concave points_mean': 0.07017,
        'symmetry_mean': 0.1812,
        'fractal_dimension_mean': 0.05667,
        'radius_se': 0.5435,
        'texture_se': 0.7339,
        'perimeter_se': 3.398,
        'area_se': 74.08,
        'smoothness_se': 0.005225,
        'compactness_se': 0.01308,
        'concavity_se': 0.0186,
        'concave points_se': 0.0134,
        'symmetry_se': 0.01389,
        'fractal_dimension_se': 0.003532,
        'radius_worst': 24.99,
        'texture_worst': 23.41,
        'perimeter_worst': 158.8,
        'area_worst': 1956.0,
        'smoothness_worst': 0.1238,
        'compactness_worst': 0.1866,
        'concavity_worst': 0.2416,
        'concave points_worst': 0.1860,
        'symmetry_worst': 0.2750,
        'fractal_dimension_worst': 0.08902
    }
    
    print("Testing prediction on Malignant sample data...")
    result = predict_tumor(sample)
    if "error" in result:
        print(f"Prediction failed: {result['error']}")
    else:
        print(f"Prediction: {result['Prediction']}")
        print(f"{result['Score Type']}: {result['Score']:.4f}")
        print(f"Threshold Used: {result['Threshold']:.4f}")
        
    print("\n----------------------------------------\n")
    
    # Example prediction using a Benign-like sample
    benign_sample = {
        'radius_mean': 13.54,
        'texture_mean': 14.36,
        'perimeter_mean': 87.46,
        'area_mean': 566.3,
        'smoothness_mean': 0.09779,
        'compactness_mean': 0.08129,
        'concavity_mean': 0.06664,
        'concave points_mean': 0.04781,
        'symmetry_mean': 0.1885,
        'fractal_dimension_mean': 0.05766,
        'radius_se': 0.2699,
        'texture_se': 0.7886,
        'perimeter_se': 2.058,
        'area_se': 23.56,
        'smoothness_se': 0.008462,
        'compactness_se': 0.0146,
        'concavity_se': 0.02387,
        'concave points_se': 0.01315,
        'symmetry_se': 0.0198,
        'fractal_dimension_se': 0.0023,
        'radius_worst': 15.11,
        'texture_worst': 19.26,
        'perimeter_worst': 99.7,
        'area_worst': 711.2,
        'smoothness_worst': 0.144,
        'compactness_worst': 0.1773,
        'concavity_worst': 0.239,
        'concave points_worst': 0.1288,
        'symmetry_worst': 0.2977,
        'fractal_dimension_worst': 0.07259
    }
    
    print("Testing prediction on Benign sample data...")
    benign_result = predict_tumor(benign_sample)
    if "error" in benign_result:
        print(f"Prediction failed: {benign_result['error']}")
    else:
        print(f"Prediction: {benign_result['Prediction']}")
        print(f"{benign_result['Score Type']}: {benign_result['Score']:.4f}")
        print(f"Threshold Used: {benign_result['Threshold']:.4f}")


Testing prediction on Malignant sample data...
Prediction: Malignant
Probability: 1.0000
Threshold Used: 0.6479

----------------------------------------

Testing prediction on Benign sample data...
Prediction: Benign
Probability: 0.0270
Threshold Used: 0.6479
